# Đọc VCG features đã trích xuất

Notebook này chỉ đọc các bảng features có sẵn trong repo. Không tải PTB, không
kết nối Drive và không tính lại median hay QRS/T. Chạy trực tiếp trên Colab hoặc
máy có pandas. Nếu chạy trong bản clone, CSV được đọc từ ổ đĩa; nếu mở riêng trên
Colab, chỉ tải ba CSV nhỏ từ GitHub.

Chọn `VERSION="v1"` cho kết quả Notebook06 hoặc `"v2"` cho T features Wavelet/TLS.
Cả hai bộ có đủ 448 dòng và cùng cách chia bệnh nhân. V2 chưa được xác nhận tốt hơn V1.

In [ ]:
from pathlib import Path
import pandas as pd

VERSION = 'v1'
if VERSION not in {'v1', 'v2'}:
    raise ValueError('VERSION phải là v1 hoặc v2.')
FEATURE_COLUMNS = [
    'QRS_Arc', 'QRS_Area', 'QRS_MaxVec', 'QRS_MaxCentroidDistance',
    'QRS_MaxVel', 'QRS_MeanVel', 'QRS_StdVel',
    'T_MaxVec', 'T_Area', 'T_MaxVel', 'T_MeanVel', 'T_StdVel',
]
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'data/features/metadata.json').is_file()), None)
base = str(repo/'data/features') if repo else 'https://raw.githubusercontent.com/NgThach/VCG/main/data/features'
train = pd.read_csv(f'{base}/{VERSION}/train_features.csv')
valid = pd.read_csv(f'{base}/{VERSION}/valid_features.csv')
test = pd.read_csv(f'{base}/{VERSION}/test_features.csv')
print('Nguồn CSV:', base)
print('Bộ feature:', VERSION)
display(train.head())

## X là 12 feature, y là nhãn MI/HC

Nhãn HC=0, MI=1. ID và tên tập không được đưa vào X.
Các giá trị thiếu được giữ NaN; chưa áp dụng cách điền thiếu hoặc loại dòng.

In [ ]:
X_train, y_train = train[FEATURE_COLUMNS].copy(), train['label'].copy()
X_valid, y_valid = valid[FEATURE_COLUMNS].copy(), valid['label'].copy()
X_test, y_test = test[FEATURE_COLUMNS].copy(), test['label'].copy()
print('X_train:', X_train.shape, '| X_valid:', X_valid.shape, '| X_test:', X_test.shape)
display(pd.DataFrame([
    {'split': name, 'rows': len(frame),
     'full_12_features': int(frame[FEATURE_COLUMNS].notna().all(axis=1).sum()),
     'missing_features': int(frame[FEATURE_COLUMNS].isna().any(axis=1).sum())}
    for name, frame in [('train', train), ('valid', valid), ('test', test)]
]))

## Những dòng cần xem lại

V1 có 7 dòng thiếu feature; V2 có 136 dòng thiếu. Bảng này vẫn giữ các dòng đó.
Đủ feature không có nghĩa đã được QC duyệt. Các cờ chất lượng/lý do thiếu nằm trong
CSV để bạn tự chọn cách xử lý cho bài toán của mình.

Để xem hoặc tách lại **hình dạng QRS/T loops**, cần tín hiệu XYZ/median và các
notebook nghiên cứu; 12 con số không đủ dựng lại loop.

In [ ]:
all_features = pd.concat([train, valid, test], ignore_index=True)
review_columns = ['record_id', 'split', 'label_name', 'feature_completeness']
review_columns += [c for c in ['qc_status', 'status', 'feature_missing_reasons', 'failure_reason']
                   if c in all_features.columns]
display(all_features.loc[all_features[FEATURE_COLUMNS].isna().any(axis=1), review_columns])